# Bronze Layer — Data Ingestion Overview

Surveys every table ingested into the **bronze_lakehouse** so you can confirm what landed and inspect it.

| | |
|---|---|
| **Lakehouse** | bronze_lakehouse |
| **AOI** | Maple Ridge, BC — 20 km radius |
| **Layers** | Planetary Computer STAC, PC collections, DataBC WFS geology |

Run top-to-bottom. The default lakehouse must be attached (`bronze_lakehouse`).

## 1. Discover all bronze tables

In [ ]:
# List every table in the default lakehouse and keep the bronze_ ones
all_tables = [t.name for t in spark.catalog.listTables()]
bronze_tables = sorted([t for t in all_tables if t.startswith("bronze_")])

print(f"Found {len(bronze_tables)} bronze tables:\n")
for t in bronze_tables:
    print(f"  - {t}")

## 2. Row counts per table

In [ ]:
# Build a summary of row counts and column counts for each bronze table
from pyspark.sql import Row

summary = []
for t in bronze_tables:
    try:
        df = spark.table(t)
        summary.append(Row(table=t, rows=df.count(), columns=len(df.columns)))
    except Exception as e:
        summary.append(Row(table=t, rows=-1, columns=-1))
        print(f"  ! {t}: {e}")

summary_df = spark.createDataFrame(summary).orderBy("table")
print(f"Total bronze tables: {len(summary)}")
print(f"Total rows across all bronze tables: {sum(r.rows for r in summary if r.rows > 0)}")
display(summary_df)

## 3. Schema + sample rows for each table

In [ ]:
# Inspect schema and a few sample rows for every bronze table
for t in bronze_tables:
    print("=" * 80)
    print(f"TABLE: {t}")
    print("=" * 80)
    try:
        df = spark.table(t)
        print(f"Rows: {df.count()}  |  Columns: {len(df.columns)}")
        df.printSchema()
        display(df.limit(5))
    except Exception as e:
        print(f"  ! Could not read {t}: {e}")
    print()

## 4. Maps — one per layer, each in its own cell, with a legend

The cell below defines `render_layer(table)`. **Each bronze layer then gets its
own cell** (one `render_layer(...)` call each) so every map renders separately
and overlapping rasters don't pile up into mush. Every map carries a **legend**
in the lower-left explaining its symbols.

- **STAC raster tables** (Sentinel-2, Sentinel-1, DEM, land cover, biomass…) →
  the **actual raster** is drawn as live tiles. The metadata bronze tables only
  store item footprints + asset names, so each item is re-fetched from the
  Planetary Computer data API to pull its `tilejson` tile layer (falling back to
  the `rendered_preview` PNG overlay). Each item's footprint is also drawn as a
  clickable outline.
- **DataBC geology tables** (quaternary, bedrock, faults) → drawn as their
  actual **GeoJSON geometries**.
- **Click any feature** (raster footprint or geology polygon) to see **every
  column** for that record in a scrollable popup.
- Red star + circle = AOI center and 20 km radius. Use the layer control
  (top-right) to toggle individual items on/off; the **legend** (bottom-left)
  maps each color/symbol to what it represents.

> Raster tiles are fetched live from `planetarycomputer.microsoft.com`, so the
> Spark host needs outbound internet. A few item fetches per table keeps it fast.


In [ ]:
# Map helpers: ensure folium, fetch real raster tiles from Planetary Computer,
# and build a "show every column" popup.
import sys, subprocess, json, requests
from urllib.parse import urlparse, parse_qs

try:
    import folium
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "folium", "-q"], check=True)
    import folium

LAT, LON, RADIUS_KM = 49.2193, -122.5984, 20
PC_STAC = "https://planetarycomputer.microsoft.com/api/stac/v1"
PC_ATTR = "Imagery &copy; Microsoft Planetary Computer"

bbox_cols = {"bbox_minx", "bbox_miny", "bbox_maxx", "bbox_maxy"}

# Distinct CSS-safe color per table (valid for Leaflet paths)
palette = [
    "blue", "green", "purple", "orange", "darkred", "teal", "darkgreen",
    "navy", "magenta", "brown", "gray", "steelblue", "crimson", "olive",
]
color_for = {t: palette[i % len(palette)] for i, t in enumerate(bronze_tables)}

# CSS gradients approximating common Planetary Computer continuous colormaps,
# so a colormap raster gets a real gradient swatch in the legend.
COLORMAP_GRADIENTS = {
    "terrain": "linear-gradient(to right,#333366,#1f9b6c,#f2e09a,#b8763d,#ffffff)",
    "viridis": "linear-gradient(to right,#440154,#3b528b,#21918c,#5ec962,#fde725)",
    "magma":   "linear-gradient(to right,#000004,#3b0f70,#8c2981,#de4968,#fcfdbf)",
    "inferno": "linear-gradient(to right,#000004,#420a68,#932667,#dd513a,#fcffa4)",
    "plasma":  "linear-gradient(to right,#0d0887,#6a00a8,#b12a90,#e16462,#f0f921)",
    "cividis": "linear-gradient(to right,#00224e,#35456c,#666970,#9d9540,#fee838)",
    "gray":    "linear-gradient(to right,#000000,#ffffff)",
    "greys":   "linear-gradient(to right,#ffffff,#000000)",
    "rdylgn":  "linear-gradient(to right,#d73027,#fee08b,#1a9850)",
    "spectral":"linear-gradient(to right,#d53e4f,#fee08b,#99d594,#3288bd)",
    "rdbu":    "linear-gradient(to right,#b2182b,#f7f7f7,#2166ac)",
}

# A few items per raster table keeps the live tile fetch fast.
MAX_RASTER_ITEMS = 5
MAX_VECTOR_FEATURES = 1000


def fetch_item(collection, item_id, timeout=30):
    """Fetch a STAC item from Planetary Computer. PC injects `tilejson` and
    `rendered_preview` assets that render the item with its collection's
    default config — no signing needed to read them."""
    url = f"{PC_STAC}/collections/{collection}/items/{item_id}"
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return r.json()


def tile_url_for_item(item, timeout=30):
    """Resolve an XYZ {z}/{x}/{y} raster tile template for a STAC item."""
    tj = (item.get("assets", {}) or {}).get("tilejson")
    if not tj or not tj.get("href"):
        return None
    try:
        doc = requests.get(tj["href"], timeout=timeout).json()
        tiles = doc.get("tiles") or []
        return tiles[0] if tiles else None
    except Exception:
        return None


def preview_url_for_item(item):
    """Fallback static PNG preview (covers the item footprint)."""
    a = (item.get("assets", {}) or {}).get("rendered_preview")
    return a.get("href") if a else None


def raster_class_entries(item, max_classes=14):
    """Discrete (hex_color, label) swatches from the STAC classification
    extension (e.g. ESA WorldCover, io-lulc land-cover classes)."""
    out, seen = [], set()
    for asset in (item.get("assets", {}) or {}).values():
        classes = (asset.get("classification:classes")
                   or asset.get("classification:bitfields"))
        if not classes:
            continue
        for c in classes:
            color = c.get("color-hint") or c.get("color_hint")
            if not color:
                continue
            if not str(color).startswith("#"):
                color = "#" + str(color)
            label = c.get("description") or c.get("title") or str(c.get("value"))
            key = (color.lower(), label)
            if key in seen:
                continue
            seen.add(key)
            out.append((color, label))
        if out:
            break
    extra = max(0, len(out) - max_classes)
    return out[:max_classes], extra


def raster_render_desc(item):
    """Describe how PC renders this raster from the tilejson asset query.
    Returns a legend (swatch, label) tuple, or None.
      - colormap_name -> gradient swatch + range
      - visual/RGB    -> text-only 'true/false color' line"""
    tj = (item.get("assets", {}) or {}).get("tilejson")
    if not tj or not tj.get("href"):
        return None
    q = parse_qs(urlparse(tj["href"]).query)
    cmap = (q.get("colormap_name") or [None])[0]
    if cmap:
        grad = COLORMAP_GRADIENTS.get(cmap.lower(),
                                      "linear-gradient(to right,#222,#888,#eee)")
        rng = ""
        rescale = (q.get("rescale") or [None])[0]
        if rescale:
            parts = rescale.replace("%2C", ",").split(",")
            if len(parts) == 2:
                rng = f"  ({parts[0]}–{parts[1]})"
        return ("gradient:" + grad, f"{cmap} colormap{rng}")
    assets = ",".join(q.get("assets", []))
    if "visual" in assets or "expression" in q:
        return (None, "True/false-color composite (RGB)")
    if assets:
        return (None, f"Render: {assets}")
    return None


def _esc(v):
    s = "" if v is None else str(v)
    if len(s) > 600:
        s = s[:600] + " \u2026"
    return s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def popup_html(d, title=None):
    """Scrollable HTML table showing every field in a row dict."""
    rows = "".join(
        f"<tr><td style='font-weight:600;padding:2px 6px;vertical-align:top;"
        f"white-space:nowrap'>{_esc(k)}</td>"
        f"<td style='padding:2px 6px;word-break:break-word'>{_esc(v)}</td></tr>"
        for k, v in d.items()
    )
    head = (f"<div style='font-weight:700;margin-bottom:4px'>{_esc(title)}</div>"
            if title else "")
    return ("<div style='max-height:320px;max-width:440px;overflow:auto;"
            "font-family:system-ui,sans-serif;font-size:12px'>"
            + head + "<table>" + rows + "</table></div>")


def add_legend(m, title, entries):
    """Add a fixed-position legend box to a folium map.

    `entries` is a list of (swatch, label). `swatch` is one of:
      - a CSS color string        -> filled square
      - "marker:<color>"          -> star/pin glyph
      - "ring:<color>"            -> hollow circle outline
      - "gradient:<css-gradient>" -> horizontal gradient bar (colormap rasters)
      - None                      -> text-only line (no glyph)
    """
    items = []
    for swatch, label in entries:
        if swatch is None:
            glyph = "<span style='display:inline-block;width:14px;margin-right:6px'></span>"
        elif isinstance(swatch, str) and swatch.startswith("marker:"):
            c = swatch.split(":", 1)[1]
            glyph = (f"<span style='color:{c};margin-right:6px;"
                     f"font-size:14px;line-height:14px'>&#9733;</span>")
        elif isinstance(swatch, str) and swatch.startswith("ring:"):
            c = swatch.split(":", 1)[1]
            glyph = (f"<span style='display:inline-block;width:13px;height:13px;"
                     f"border:2px solid {c};border-radius:50%;margin-right:6px'></span>")
        elif isinstance(swatch, str) and swatch.startswith("gradient:"):
            g = swatch.split(":", 1)[1]
            glyph = (f"<span style='display:inline-block;width:34px;height:13px;"
                     f"background:{g};border:1px solid #555;margin-right:6px'></span>")
        else:
            glyph = (f"<span style='display:inline-block;width:14px;height:14px;"
                     f"background:{swatch};border:1px solid #555;"
                     f"margin-right:6px'></span>")
        items.append(
            f"<div style='display:flex;align-items:center;margin:2px 0'>"
            f"{glyph}<span>{_esc(label)}</span></div>")
    html = (
        "<div style='position:fixed;bottom:22px;left:22px;z-index:9999;"
        "background:rgba(255,255,255,0.93);padding:10px 12px;border:1px solid #888;"
        "border-radius:6px;font-family:system-ui,sans-serif;font-size:12px;"
        "box-shadow:0 1px 5px rgba(0,0,0,0.3);max-width:260px;"
        "max-height:60vh;overflow:auto'>"
        f"<div style='font-weight:700;margin-bottom:5px'>{_esc(title)}</div>"
        + "".join(items) + "</div>")
    m.get_root().html.add_child(folium.Element(html))


print("Map helpers ready. Raster items per table:", MAX_RASTER_ITEMS)


In [ ]:
# render_layer(table): draw ONE bronze layer on its own map, with a legend.
# Real raster tiles for STAC tables, GeoJSON for geology tables, every-column
# popups on click. Each call below renders into its own cell output.

def new_map():
    m = folium.Map(location=[LAT, LON], zoom_start=11, tiles="CartoDB positron")
    folium.Marker(
        [LAT, LON], tooltip="AOI center — Maple Ridge, BC",
        icon=folium.Icon(color="red", icon="star"),
    ).add_to(m)
    folium.Circle(
        [LAT, LON], radius=RADIUS_KM * 1000, color="red",
        fill=False, weight=2, tooltip=f"{RADIUS_KM} km AOI",
    ).add_to(m)
    return m


def render_layer(t):
    """Render a single bronze table as its own map (with legend).
    Safe to call for a table that doesn't exist — it just prints a note."""
    if t not in set(bronze_tables):
        print(f"  - {t}: not found in bronze tables — skipped")
        return
    color = color_for.get(t, "blue")
    try:
        df = spark.table(t)
    except Exception as e:
        print(f"  ! skip {t}: {e}")
        return
    colset = set(df.columns)
    m = new_map()
    rendered = 0
    legend = [("marker:red", "AOI center — Maple Ridge, BC"),
              ("ring:red", f"{RADIUS_KM} km AOI radius")]

    if bbox_cols.issubset(colset):
        # STAC raster table — draw the actual raster per item + clickable footprint
        rows = df.limit(MAX_RASTER_ITEMS).collect()
        class_entries, class_extra = [], 0   # discrete land-cover class swatches
        render_desc = None                   # colormap / RGB description
        for r in rows:
            d = r.asDict()
            minx, miny = d.get("bbox_minx"), d.get("bbox_miny")
            maxx, maxy = d.get("bbox_maxx"), d.get("bbox_maxy")
            item_id, collection = d.get("item_id"), d.get("collection")

            tile_url = preview = None
            if item_id and collection:
                try:
                    item = fetch_item(collection, item_id)
                    tile_url = tile_url_for_item(item)
                    preview = preview_url_for_item(item)
                    if not class_entries:
                        class_entries, class_extra = raster_class_entries(item)
                    if render_desc is None:
                        render_desc = raster_render_desc(item)
                except Exception as e:
                    print(f"    ! {t} {item_id}: {e.__class__.__name__}")

            if tile_url:
                folium.TileLayer(
                    tiles=tile_url, attr=PC_ATTR, name=str(item_id),
                    overlay=True, control=True, opacity=0.9,
                ).add_to(m)
                rendered += 1
            elif preview and None not in (minx, miny, maxx, maxy):
                folium.raster_layers.ImageOverlay(
                    image=preview, bounds=[[miny, minx], [maxy, maxx]],
                    opacity=0.9, name=str(item_id),
                ).add_to(m)
                rendered += 1

            if None not in (minx, miny, maxx, maxy):
                folium.Rectangle(
                    bounds=[[miny, minx], [maxy, maxx]],
                    color=color, weight=2, fill=False,
                    popup=folium.Popup(popup_html(d, title=item_id), max_width=460),
                    tooltip=str(item_id or t),
                ).add_to(m)
        legend.append(("ring:" + color, "Item footprint (click for details)"))
        # Fold the raster's own coloring into the legend.
        if class_entries:
            legend.append((None, "Raster classes:"))
            legend.extend(class_entries)
            if class_extra:
                legend.append((None, f"… +{class_extra} more class(es)"))
        elif render_desc:
            legend.append((None, "Raster coloring:"))
            legend.append(render_desc)
        else:
            legend.append((color, "Live raster tiles (toggle top-right)"))
        label = f"{rendered} raster item(s)"

    elif "geometry_json" in colset:
        # Geology table — draw real GeoJSON geometries with full-detail popups
        rows = df.limit(MAX_VECTOR_FEATURES).collect()
        for r in rows:
            d = r.asDict()
            gj = d.get("geometry_json")
            if not gj:
                continue
            try:
                geom = json.loads(gj)
            except Exception:
                continue
            attrs = {k: v for k, v in d.items() if k != "geometry_json"}
            folium.GeoJson(
                geom,
                style_function=lambda x, c=color: {
                    "color": c, "weight": 2, "fillColor": c, "fillOpacity": 0.25,
                },
                popup=folium.Popup(popup_html(attrs, title=t), max_width=460),
                tooltip=t,
            ).add_to(m)
            rendered += 1
        legend.append((color, "Geology features (click for details)"))
        label = f"{rendered} geometry feature(s)"

    else:
        print(f"  - {t}: no spatial columns (bbox_* or geometry_json) — skipped")
        return

    folium.LayerControl(collapsed=False).add_to(m)
    add_legend(m, t, legend)
    print(f"  + {t}: {label} ({color})")

    header = (f"<h3 style='font-family:system-ui,sans-serif;margin:14px 0 4px'>"
              f"{t} <span style='font-weight:400;color:#666;font-size:13px'>"
              f"— {label}</span></h3>")
    try:
        displayHTML(header + m._repr_html_())
    except NameError:
        display(m)


print("render_layer() ready — one map per cell below, each with a legend.")
print("Tables available:", ", ".join(bronze_tables))


In [ ]:
# Sentinel-2 L2A — optical imagery
render_layer("bronze_sentinel_2_l2a")

In [ ]:
# Sentinel-1 RTC — radar (SAR)
render_layer("bronze_sentinel_1_rtc")

In [ ]:
# ALOS PALSAR mosaic — L-band radar
render_layer("bronze_alos_palsar_mosaic")

In [ ]:
# Copernicus DEM GLO-30 — 30 m elevation
render_layer("bronze_cop_dem_glo_30")

In [ ]:
# ESA WorldCover — land cover
render_layer("bronze_esa_worldcover")

In [ ]:
# IO LULC 9-class — land use / land cover
render_layer("bronze_io_lulc_9_class")

In [ ]:
# HGB — above/below-ground biomass
render_layer("bronze_hgb")

In [ ]:
# BC Quaternary / surficial geology
render_layer("bronze_bc_quaternary_geology")

In [ ]:
# BC bedrock geology
render_layer("bronze_bc_bedrock_geology")

In [ ]:
# BC geological faults
render_layer("bronze_bc_geological_faults")